In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import xgboost as xgb
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, matthews_corrcoef
import joblib
import os

In [2]:
from sklearn.datasets import load_breast_cancer

breast_cancer = load_breast_cancer()
data_df = pd.DataFrame(breast_cancer.data, columns=breast_cancer.feature_names)
data_df['target'] = breast_cancer.target

print(f"Dataset shape: {data_df.shape}")
print(f"Features: {data_df.shape[1] - 1}")
print(f"Samples: {data_df.shape[0]}")
print(f"Target classes: {breast_cancer.target_names}")

# Display basic info
data_df.info()

Dataset shape: (569, 31)
Features: 30
Samples: 569
Target classes: ['malignant' 'benign']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 31 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   mean radius              569 non-null    float64
 1   mean texture             569 non-null    float64
 2   mean perimeter           569 non-null    float64
 3   mean area                569 non-null    float64
 4   mean smoothness          569 non-null    float64
 5   mean compactness         569 non-null    float64
 6   mean concavity           569 non-null    float64
 7   mean concave points      569 non-null    float64
 8   mean symmetry            569 non-null    float64
 9   mean fractal dimension   569 non-null    float64
 10  radius error             569 non-null    float64
 11  texture error            569 non-null    float64
 12  perimeter error          569 non-null    flo

In [3]:
print("Target distribution:")
print(data_df['target'].value_counts())
print(f"Class balance: {data_df['target'].value_counts(normalize=True)}")
print(f"0: {breast_cancer.target_names[0]}, 1: {breast_cancer.target_names[1]}")

Target distribution:
target
1    357
0    212
Name: count, dtype: int64
Class balance: target
1    0.627417
0    0.372583
Name: proportion, dtype: float64
0: malignant, 1: benign


In [5]:
# Prepare features and target
X = data_df.drop('target', axis=1)
y = data_df['target']

# --- Custom Data Splitting Strategy ---
# 1. Keep random 50 records for the fixed test set.
# 2. Split the remaining 519 records into training and validation datasets.

n_total_records = len(X)
n_fixed_test_records = 50

if n_fixed_test_records >= n_total_records:
    raise ValueError("Number of fixed test records must be less than total records.")

# Step 1: Split out 50 records for the fixed test set
X_remaining, X_fixed_test_raw, y_remaining, y_fixed_test_raw = train_test_split(
    X, y, test_size=n_fixed_test_records, random_state=42, stratify=y
)

print(f"Original dataset size: {n_total_records}")
print(f"Fixed test set size (raw): {len(X_fixed_test_raw)}")
print(f"Remaining records for train/validation: {len(X_remaining)}")

# Save the raw fixed test set to a CSV file
os.makedirs('model', exist_ok=True) # Ensure 'data' directory exists
fixed_test_df_raw = X_fixed_test_raw.copy()
fixed_test_df_raw['target'] = y_fixed_test_raw
fixed_test_df_raw.to_csv("model/fixed_breast_cancer_test_data_raw.csv", index=False)
print(f"Saved {n_fixed_test_records} raw fixed test records to model/fixed_breast_cancer_test_data_raw.csv")


# Step 2: Split the remaining 519 records into training and validation sets
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_remaining, y_remaining, test_size=0.2, random_state=42, stratify=y_remaining
)

print(f"Training set size (raw): {len(X_train_raw)} records")
print(f"Validation set size (raw): {len(X_val_raw)} records")



# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_val_scaled = scaler.transform(X_val_raw)

print(f"Training set: {X_train_scaled.shape}")
print(f"Validation set: {X_val_scaled.shape}")

Original dataset size: 569
Fixed test set size (raw): 50
Remaining records for train/validation: 519
Saved 50 raw fixed test records to model/fixed_breast_cancer_test_data_raw.csv
Training set size (raw): 415 records
Validation set size (raw): 104 records
Training set: (415, 30)
Validation set: (104, 30)


In [6]:
# Function to calculate all metrics
def calculate_all_metrics(y_val, y_pred, y_pred_probability):
    metrics = {
        'Accuracy': accuracy_score(y_val, y_pred),
        'AUC Score': roc_auc_score(y_val, y_pred_probability),
        'Precision': precision_score(y_val, y_pred),
        'Recall': recall_score(y_val, y_pred),
        'F1 Score': f1_score(y_val, y_pred),
        'MCC Score': matthews_corrcoef(y_val, y_pred)
    }
    
    return metrics

In [7]:
#Logistic Regression

# Instantiate model
logisticRegressionModel = LogisticRegression(random_state=42, solver='liblinear')

# Fit the model
logisticRegressionModel.fit(X_train_scaled, y_train)

# Make predictions on the test data
y_pred_logistic_regression = logisticRegressionModel.predict(X_val_scaled)

# Obtain positive class probability predictions
y_pred_probability_logistic_regression = logisticRegressionModel.predict_proba(X_val_scaled)[:, 1]

# Calculate evaluation metrics

metrics = calculate_all_metrics(y_val, y_pred_logistic_regression, y_pred_probability_logistic_regression)

print("--- Logistic Regression Model Evaluation ---")
print(f"Accuracy: {metrics['Accuracy']:.4f}")
print(f"AUC Score: {metrics['AUC Score']:.4f}")
print(f"Precision: {metrics['Precision']:.4f}")
print(f"Recall: {metrics['Recall']:.4f}")
print(f"F1 Score: {metrics['F1 Score']:.4f}")
print(f"MCC Score: {metrics['MCC Score']:.4f}")

logistic_regression_metrics = {
    'Model': 'Logistic Regression',
    'Accuracy': metrics['Accuracy'],
    'AUC Score': metrics['AUC Score'],
    'Precision': metrics['Precision'],
    'Recall': metrics['Recall'],
    'F1 Score': metrics['F1 Score'],
    'MCC': metrics['MCC Score']
}

print("\nLogistic Regression Metrics stored:")
print(logistic_regression_metrics)

joblib.dump(logisticRegressionModel, f"model/logisticRegressionModel.pkl")

--- Logistic Regression Model Evaluation ---
Accuracy: 0.9808
AUC Score: 0.9846
Precision: 0.9846
Recall: 0.9846
F1 Score: 0.9846
MCC Score: 0.9590

Logistic Regression Metrics stored:
{'Model': 'Logistic Regression', 'Accuracy': 0.9807692307692307, 'AUC Score': 0.9846153846153846, 'Precision': 0.9846153846153847, 'Recall': 0.9846153846153847, 'F1 Score': 0.9846153846153847, 'MCC': 0.958974358974359}


['model/logisticRegressionModel.pkl']

In [8]:
# Decision Tree Classifier

# Instantiate model
decisionTreeClassifierModel = DecisionTreeClassifier(random_state=42)

# Fit the model
decisionTreeClassifierModel.fit(X_train_scaled, y_train)

# Make predictions on the test data
y_pred_decision_tree_classifier = decisionTreeClassifierModel.predict(X_val_scaled)

# Obtain positive class probability predictions
y_pred_probability_decision_tree_classifier = decisionTreeClassifierModel.predict_proba(X_val_scaled)[:, 1]

# Calculate evaluation metrics
metrics = calculate_all_metrics(y_val, y_pred_decision_tree_classifier, y_pred_probability_decision_tree_classifier)


print("--- Decision Tree Classifier Model Evaluation ---")
print(f"Accuracy: {metrics['Accuracy']:.4f}")
print(f"AUC Score: {metrics['AUC Score']:.4f}")
print(f"Precision: {metrics['Precision']:.4f}")
print(f"Recall: {metrics['Recall']:.4f}")
print(f"F1 Score: {metrics['F1 Score']:.4f}")
print(f"MCC Score: {metrics['MCC Score']:.4f}")

decision_tree_metrics = {
    'Model': 'Decision Tree Classifier',
    'Accuracy': metrics['Accuracy'],
    'AUC Score': metrics['AUC Score'],
    'Precision': metrics['Precision'],
    'Recall': metrics['Recall'],
    'F1 Score': metrics['F1 Score'],
    'MCC': metrics['MCC Score']
}

print("\nDecision Tree Classifier Metrics stored:")
print(decision_tree_metrics)

joblib.dump(decisionTreeClassifierModel, f"model/decisionTreeClassifierModel.pkl")

--- Decision Tree Classifier Model Evaluation ---
Accuracy: 0.9423
AUC Score: 0.9385
Precision: 0.9538
Recall: 0.9538
F1 Score: 0.9538
MCC Score: 0.8769

Decision Tree Classifier Metrics stored:
{'Model': 'Decision Tree Classifier', 'Accuracy': 0.9423076923076923, 'AUC Score': 0.9384615384615386, 'Precision': 0.9538461538461539, 'Recall': 0.9538461538461539, 'F1 Score': 0.9538461538461539, 'MCC': 0.8769230769230769}


['model/decisionTreeClassifierModel.pkl']

In [9]:
# KNN Classifier

# Instantiate model
knnModel = KNeighborsClassifier(n_neighbors=5)

# Fit the model
knnModel.fit(X_train_scaled, y_train)

# Make predictions on the test data
y_pred_knn = knnModel.predict(X_val_scaled)

# Obtain positive class probability predictions
y_pred_probability_knn = knnModel.predict_proba(X_val_scaled)[:, 1]

# Calculate evaluation metrics
metrics = calculate_all_metrics(y_val, y_pred_knn, y_pred_probability_knn)

print("--- K-Nearest Neighbor Classifier Model Evaluation ---")
print(f"Accuracy: {metrics['Accuracy']:.4f}")
print(f"AUC Score: {metrics['AUC Score']:.4f}")
print(f"Precision: {metrics['Precision']:.4f}")
print(f"Recall: {metrics['Recall']:.4f}")
print(f"F1 Score: {metrics['F1 Score']:.4f}")
print(f"MCC Score: {metrics['MCC Score']:.4f}")

knn_metrics = {
    'Model': 'K-Nearest Neighbor Classifier',
    'Accuracy': metrics['Accuracy'],
    'AUC Score': metrics['AUC Score'],
    'Precision': metrics['Precision'],
    'Recall': metrics['Recall'],
    'F1 Score': metrics['F1 Score'],
    'MCC': metrics['MCC Score']
}

print("\nK-Nearest Neighbor Classifier Metrics stored:")
print(knn_metrics)

joblib.dump(knnModel, f"model/knnModel.pkl")

--- K-Nearest Neighbor Classifier Model Evaluation ---
Accuracy: 0.9808
AUC Score: 0.9850
Precision: 0.9701
Recall: 1.0000
F1 Score: 0.9848
MCC Score: 0.9594

K-Nearest Neighbor Classifier Metrics stored:
{'Model': 'K-Nearest Neighbor Classifier', 'Accuracy': 0.9807692307692307, 'AUC Score': 0.9850098619329388, 'Precision': 0.9701492537313433, 'Recall': 1.0, 'F1 Score': 0.9848484848484849, 'MCC': 0.9593737592566562}


['model/knnModel.pkl']

In [10]:
# Naive Bayes Classifier

# Instantiate model
naiveBayesModel = GaussianNB()

# Fit the model
naiveBayesModel.fit(X_train_scaled, y_train)

# Make predictions on the test data
y_pred_naive_bayes = naiveBayesModel.predict(X_val_scaled)

# Obtain positive class probability predictions
y_pred_probability_naive_bayes = naiveBayesModel.predict_proba(X_val_scaled)[:, 1]

# Calculate evaluation metrics
metrics = calculate_all_metrics(y_val, y_pred_naive_bayes, y_pred_probability_naive_bayes)


print("--- Naive Bayes Classifier Model Evaluation ---")
print(f"Accuracy: {metrics['Accuracy']:.4f}")
print(f"AUC Score: {metrics['AUC Score']:.4f}")
print(f"Precision: {metrics['Precision']:.4f}")
print(f"Recall: {metrics['Recall']:.4f}")
print(f"F1 Score: {metrics['F1 Score']:.4f}")
print(f"MCC Score: {metrics['MCC Score']:.4f}")


naive_bayes_metrics = {
    'Model': 'Naive Bayes Classifier',
    'Accuracy': metrics['Accuracy'],
    'AUC Score': metrics['AUC Score'],
    'Precision': metrics['Precision'],
    'Recall': metrics['Recall'],
    'F1 Score': metrics['F1 Score'],
    'MCC': metrics['MCC Score']
}

print("\nNaive Bayes Classifier Metrics stored:")
print(naive_bayes_metrics)

joblib.dump(naiveBayesModel, f"model/naiveBayesModel.pkl")

--- Naive Bayes Classifier Model Evaluation ---
Accuracy: 0.9327
AUC Score: 0.9866
Precision: 0.9394
Recall: 0.9538
F1 Score: 0.9466
MCC Score: 0.8559

Naive Bayes Classifier Metrics stored:
{'Model': 'Naive Bayes Classifier', 'Accuracy': 0.9326923076923077, 'AUC Score': 0.9865877712031559, 'Precision': 0.9393939393939394, 'Recall': 0.9538461538461539, 'F1 Score': 0.9465648854961832, 'MCC': 0.8558520444308153}


['model/naiveBayesModel.pkl']

In [11]:
# Random Forest Classifier

# Instantiate model
randomForestmodel = RandomForestClassifier(random_state=42)

# Fit the model
randomForestmodel.fit(X_train_scaled, y_train)

# Make predictions on the test data
y_pred_random_forest = randomForestmodel.predict(X_val_scaled)

# Obtain positive class probability predictions
y_pred_probability_random_forest = randomForestmodel.predict_proba(X_val_scaled)[:, 1]

# Calculate evaluation metrics
metrics = calculate_all_metrics(y_val, y_pred_random_forest, y_pred_probability_random_forest)


print("--- Random Forest Classifier Model Evaluation ---")
print(f"Accuracy: {metrics['Accuracy']:.4f}")
print(f"AUC Score: {metrics['AUC Score']:.4f}")
print(f"Precision: {metrics['Precision']:.4f}")
print(f"Recall: {metrics['Recall']:.4f}")
print(f"F1 Score: {metrics['F1 Score']:.4f}")
print(f"MCC Score: {metrics['MCC Score']:.4f}")


random_forest_metrics = {
    'Model': 'Random Forest Classifier',
    'Accuracy': metrics['Accuracy'],
    'AUC Score': metrics['AUC Score'],
    'Precision': metrics['Precision'],
    'Recall': metrics['Recall'],
    'F1 Score': metrics['F1 Score'],
    'MCC': metrics['MCC Score']
}

print("\nRandom Forest Classifier Metrics stored:")
print(random_forest_metrics)

joblib.dump(randomForestmodel, f"model/randomForestmodel.pkl")

--- Random Forest Classifier Model Evaluation ---
Accuracy: 0.9615
AUC Score: 0.9905
Precision: 0.9841
Recall: 0.9538
F1 Score: 0.9688
MCC Score: 0.9195

Random Forest Classifier Metrics stored:
{'Model': 'Random Forest Classifier', 'Accuracy': 0.9615384615384616, 'AUC Score': 0.9905325443786983, 'Precision': 0.9841269841269841, 'Recall': 0.9538461538461539, 'F1 Score': 0.96875, 'MCC': 0.9195402465724164}


['model/randomForestmodel.pkl']

In [12]:
# XGBoost Classifier

# Instantiate model
xgBoostModel = XGBClassifier(random_state=42, eval_metric='logloss') # use_label_encoder is removed as it is deprecated and no longer needed

# Fit the model
xgBoostModel.fit(X_train_scaled, y_train)

# Make predictions on the test data
y_pred_xgboost = xgBoostModel.predict(X_val_scaled)

# Obtain positive class probability predictions
y_pred_probability_xgboost = xgBoostModel.predict_proba(X_val_scaled)[:, 1]

# Calculate evaluation metrics
metrics = calculate_all_metrics(y_val, y_pred_xgboost, y_pred_probability_xgboost)


print("--- XGBoost Classifier Model Evaluation ---")
print(f"Accuracy: {metrics['Accuracy']:.4f}")
print(f"AUC Score: {metrics['AUC Score']:.4f}")
print(f"Precision: {metrics['Precision']:.4f}")
print(f"Recall: {metrics['Recall']:.4f}")
print(f"F1 Score: {metrics['F1 Score']:.4f}")
print(f"MCC Score: {metrics['MCC Score']:.4f}")


xgboost_metrics = {
    'Model': 'XGBoost Classifier',
    'Accuracy': metrics['Accuracy'],
    'AUC Score': metrics['AUC Score'],
    'Precision': metrics['Precision'],
    'Recall': metrics['Recall'],
    'F1 Score': metrics['F1 Score'],
    'MCC': metrics['MCC Score']
}

print("\nXGBoost Classifier Metrics stored:")
print(xgboost_metrics)

joblib.dump(xgBoostModel, f"model/xgBoostModel.pkl")

--- XGBoost Classifier Model Evaluation ---
Accuracy: 0.9615
AUC Score: 0.9893
Precision: 0.9692
Recall: 0.9692
F1 Score: 0.9692
MCC Score: 0.9179

XGBoost Classifier Metrics stored:
{'Model': 'XGBoost Classifier', 'Accuracy': 0.9615384615384616, 'AUC Score': 0.9893491124260355, 'Precision': 0.9692307692307692, 'Recall': 0.9692307692307692, 'F1 Score': 0.9692307692307692, 'MCC': 0.9179487179487179}


['model/xgBoostModel.pkl']

In [13]:
# Saving scalar.py so that it can be used to preprocess the test data in Streamlit app
joblib.dump(scaler, "model/scaler.py")

['model/scaler.py']

In [14]:
# Comparison

# Collect all metrics dictionaries into a list
all_metrics = [
    logistic_regression_metrics,
    decision_tree_metrics,
    knn_metrics,
    naive_bayes_metrics,
    random_forest_metrics,
    xgboost_metrics
]

# Create a DataFrame from the list of dictionaries
metrics_df = pd.DataFrame(all_metrics)

# Display the comparison table, formatting numerical columns for better readability
# Exclude the 'Model' column from formatting to keep it as string
formatted_metrics_df = metrics_df.copy()
for col in formatted_metrics_df.columns:
    if col != 'Model':
        formatted_metrics_df[col] = formatted_metrics_df[col].apply(lambda x: f"{x:.4f}")

print("\n--- Model Performance Comparison Table ---")
print(formatted_metrics_df.to_string())


--- Model Performance Comparison Table ---
                           Model Accuracy AUC Score Precision  Recall F1 Score     MCC
0            Logistic Regression   0.9808    0.9846    0.9846  0.9846   0.9846  0.9590
1       Decision Tree Classifier   0.9423    0.9385    0.9538  0.9538   0.9538  0.8769
2  K-Nearest Neighbor Classifier   0.9808    0.9850    0.9701  1.0000   0.9848  0.9594
3         Naive Bayes Classifier   0.9327    0.9866    0.9394  0.9538   0.9466  0.8559
4       Random Forest Classifier   0.9615    0.9905    0.9841  0.9538   0.9688  0.9195
5             XGBoost Classifier   0.9615    0.9893    0.9692  0.9692   0.9692  0.9179
